In [19]:
import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import seaborn as sns

In [20]:
# Load dataset handling the latin-1 text encoding variation safely
df = pd.read_csv("AviationData.csv", encoding='latin-1', low_memory=False)

# Inspect missing values (NaNs) and column datatypes
df.info()

# Print out general summary statistics for the numerical metrics
print("\n SUMMARY STATISTICS ")
df.describe(include='all')

<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  str    
 7   Longitude               34373 non-null  str    
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str    
 

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
count,88889,88889,88889,88889,88837,88663,34382,34373,50132,52704,...,82697,16648,77488.000000,76379.000000,76956.000000,82977.000000,84397,61724,82505,75118
unique,87951,2,88863,14782,27758,219,25589,27154,10374,24870,...,26,13590,NaN,NaN,NaN,NaN,4,12,17074,2924
top,20001214X45071,Accident,ERA22LA103,1982-05-16,"ANCHORAGE, AK",United States,332739N,0112457W,NONE,Private,...,Personal,Pilot,NaN,NaN,NaN,NaN,VMC,Landing,Probable Cause,25-09-2020
freq,3,85015,2,25,434,82248,19,24,1488,240,...,49448,258,NaN,NaN,NaN,NaN,77303,15428,61754,17019
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.647855,0.279881,0.357061,5.325440,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,5.485960,1.544084,2.235625,27.913634,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.000000,0.000000,0.000000,1.000000,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.000000,0.000000,0.000000,2.000000,NaN,NaN,NaN,NaN


In [21]:
# 1. Convert Date column to datetime format
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')
df['Year'] = df['Event.Date'].dt.year

# 2. Filter dataset matching criteria: Airplanes only, Professional builds, and Last 40 Years (Post-1983)
df_cleaned = df[
    (df['Aircraft.Category'].str.upper() == 'AIRPLANE') &
    (df['Amateur.Built'].str.upper() == 'NO') &
    (df['Year'] >= 1983) & (df['Year'] <= 2023)
].copy()

print(f"Remaining observations after primary filtering: {df_cleaned.shape[0]}")

Remaining observations after primary filtering: 21447


In [22]:
# Impute NaNs in injury metrics to 0 so mathematical transformations don't result in NaN
for col in ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']:
    if col in df_cleaned.columns:
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce').fillna(0)

# 1. Create a metric for both fatal and serious injuries
df_cleaned['Severe.Injuries'] = df_cleaned['Total.Fatal.Injuries'] + df_cleaned['Total.Serious.Injuries']

# Estimate total capacity on board (Sum of all injury states + uninjured)
df_cleaned['Estimated.Total.People'] = (
    df_cleaned['Severe.Injuries'] + 
    df_cleaned['Total.Minor.Injuries'] + 
    df_cleaned['Total.Uninjured']
)

# Likelihood rate calculation 
df_cleaned['Severe.Injury.Rate'] = np.where(
    df_cleaned['Estimated.Total.People'] > 0,
    df_cleaned['Severe.Injuries'] / df_cleaned['Estimated.Total.People'],
    0
)

# 2. Derived column tracking whether an aircraft was completely destroyed or not
df_cleaned['Is.Destroyed'] = df_cleaned['Aircraft.damage'].str.upper() == 'DESTROYED'

In [23]:
# Cleaning by converting to string upper case and removing spaces
df_cleaned['Make'] = df_cleaned['Make'].astype(str).str.upper().str.strip()

# Calculate incident frequencies per manufacture brand
make_counts = df_cleaned['Make'].value_counts()

# Apply threshold of keeping brands with at least 50 historical incidents
valid_makes = make_counts[make_counts >= 50].index
df_cleaned = df_cleaned[df_cleaned['Make'].isin(valid_makes)].copy()

print(f"The reliable Aircraft Makes remaining of (Threshold >= 50) are: {len(df_cleaned['Make'].unique())}")

The reliable Aircraft Makes remaining of (Threshold >= 50) are: 36


In [24]:
# 1. Drop records where the aircraft model is unknown
df_cleaned = df_cleaned.dropna(subset=['Model'])

# 2. Clean formatting variations on model strings
df_cleaned['Model'] = df_cleaned['Model'].astype(str).str.upper().str.strip()

# 3. Create a unique identifier key for a given plane type combining Make + Model
df_cleaned['Unique.Plane.ID'] = df_cleaned['Make'] + " " + df_cleaned['Model']

print(f"Total distinct plane models are: {len(df_cleaned['Unique.Plane.ID'].unique())}")

Total distinct plane models are: 2153


In [25]:
other_cols = ['Engine.Type', 'Weather.Condition', 'Number.of.Engines', 'Purpose.of.flight', 'Broad.phase.of.flight']

# Clean string text fields, standardize casing, and handle unknown placeholders explicitly
for col in other_cols:
    if col in df_cleaned.columns:
        if df_cleaned[col].dtype == 'object':
            df_cleaned[col] = df_cleaned[col].astype(str).str.upper().str.strip()
            # Replace missing data clearly with UNKNOWN 
            df_cleaned[col] = df_cleaned[col].replace(['NAN', 'UNK', '', 'NONE'], 'UNKNOWN')
        else:
            df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

print("\n Cleaned Dataset Summary ")
df_cleaned[other_cols].head()


 Cleaned Dataset Summary 


,Engine.Type,Weather.Condition,Number.of.Engines,Purpose.of.flight,Broad.phase.of.flight
4150,NaN,NaN,4.0,NaN,NaN
4171,NaN,NaN,1.0,NaN,NaN
4285,NaN,NaN,2.0,NaN,NaN
6760,NaN,NaN,3.0,NaN,NaN
6806,NaN,NaN,1.0,NaN,NaN


In [26]:
# 1. Identify original columns that have more than 20,000 non-nulls in 'df'
original_non_nulls = df.notna().sum()
valid_original_cols = original_non_nulls[original_non_nulls > 20000].index.tolist()


custom_engineered_cols = [
    'Severe.Injuries', 'Estimated.Total.People', 
    'Severe.Injury.Rate', 'Is.Destroyed', 'Unique.Plane.ID'
]

# 3. Combine both lists 
final_columns_to_keep = [col for col in (valid_original_cols + custom_engineered_cols) if col in df_cleaned.columns]

# 4. Apply column filter
df_cleaned = df_cleaned[final_columns_to_keep].copy()

# 5. Verify the remaining structure
print(f" Columns remaining after the threshold filter: {df_cleaned.shape[1]}")
df_cleaned.columns.tolist()

 Columns remaining after the threshold filter: 35


['Event.Id',
 'Investigation.Type',
 'Accident.Number',
 'Event.Date',
 'Location',
 'Country',
 'Latitude',
 'Longitude',
 'Airport.Code',
 'Airport.Name',
 'Injury.Severity',
 'Aircraft.damage',
 'Aircraft.Category',
 'Registration.Number',
 'Make',
 'Model',
 'Amateur.Built',
 'Number.of.Engines',
 'Engine.Type',
 'FAR.Description',
 'Purpose.of.flight',
 'Total.Fatal.Injuries',
 'Total.Serious.Injuries',
 'Total.Minor.Injuries',
 'Total.Uninjured',
 'Weather.Condition',
 'Broad.phase.of.flight',
 'Report.Status',
 'Publication.Date',
 'Year',
 'Severe.Injuries',
 'Estimated.Total.People',
 'Severe.Injury.Rate',
 'Is.Destroyed',
 'Unique.Plane.ID']

In [27]:
# Save the cleaned dataset to a CSV file 
df_cleaned.to_csv("cleaned_aviation_data.csv", index=False)